In [ ]:
import os
import nibabel as nib
import numpy as np 
!pip install nipype
np.set_printoptions(precision=2, suppress=True)
from nipype.interfaces.fsl import Reorient2Std
from nipype.interfaces.fsl import RobustFOV
from nipype.interfaces.fsl import FLIRT
from nipype.interfaces.fsl import BET
from nipype.interfaces.ants.segmentation import BrainExtraction
from nipype.interfaces.ants import MeasureImageSimilarity
#from fsl.utils.image.resample import resampleToPixdims
from nipype.interfaces.ants import N4BiasFieldCorrection
import nipype.interfaces.dipy as dipy
import matplotlib.pyplot as plt
from matplotlib.image import imread
import matplotlib.image as mpimg
from pathlib import Path
import scipy.ndimage as ndi
#import cv2
import openpyxl

In [ ]:
FLIRT.inputs_help()

In [ ]:
#Functions checking

def convertTuple(tup):
        # initialize an empty string
    str = ''
    for item in tup:
        str = str + item
    return str

def OrientationCheck(path_nii):
    path=path_nii
    vol=nib.load(path)
    # info=vol.get_fdata()
    # hdr=vol.header
    # pixDim_all=hdr['pixdim']
    # pixDim=(pixDim_all[1], pixDim_all[2], pixDim_all[3])
    # matrixDim=vol.shape
    orientation_label=nib.aff2axcodes(vol.affine)
    str_orientation = convertTuple(orientation_label)
    return str_orientation

def nifti2jpg_old(nii_folder,pat_num,mod,step):
    root_path=r'DATA_ROOT/checking_preProcessing' #Write the path where you want to store the jpg images
    nii_slice_folder=os.path.join(root_path,pat_num,pat_num+mod+'_' +step+'_slice'+'.nii.gz')
    jpg_folder=os.path.join(root_path,pat_num,pat_num+ mod+'_' +step+'_slice.jpg')
    nii_img=nib.load(nii_folder)
    info_nii=nii_img.get_fdata()
    my_slice=info_nii[:,:,int(np.round(info_nii.shape[0]/6))]
    one_slice_nii=nib.Nifti1Image(my_slice, nii_img.affine, nii_img.header)
    nib.save(one_slice_nii,pat_num+mod+'_' +step+'_slice'+'.nii.gz')
    jpg_img=nib.load(nii_slice_folder).get_fdata()/8
  
    status=cv2.imwrite(jpg_folder,jpg_img)
    print(status)
    print(jpg_folder)
    print(nii_slice_folder)
    return jpg_folder



import skimage.io as io
def nifti2jpg(nii_folder,pat_num,mod,step):
    root_path=r'DATA_ROOT/checking_results_v2' #Write the path where you want to store the jpg images
    nii_slice_folder=os.path.join(root_path,pat_num,pat_num+mod+'_' +step+'_slice'+'.nii.gz')
    jpg_folder=os.path.join(root_path,mod,pat_num,pat_num+ mod+'_' +step+'_slice.jpg')
    nii_img=nib.load(nii_folder)
    info_nii=nii_img.get_fdata()
    my_slice=info_nii[:,:,int(np.round(info_nii.shape[0]/6))]
    io.imsave(jpg_folder,my_slice)
    return jpg_folder

def BiasFieldCheck(pathLikeBefore,pathLikeAfter,pat,mod):
    filenameBefore=pathLikeBefore #Aqui va el nombre del archivo before bias field
    filenameAfter=pathLikeAfter

    #Read the figure before correction
    dataBefore=imread(filenameBefore)
    #Read the figure after correction
    dataAfter=imread(filenameAfter)


    #Obtain the position of the pixel with the maximum value BEFORE correction
    maxrows = np.argmax(dataBefore,axis=0)
    maxrowsVal = dataBefore
    
    #Obtain the position of the pixel with the maximum value AFTER correction
    maxrows_after = np.argmax(dataAfter,axis=0)
    maxrowsVal_after = dataAfter
    
    ########## BEGIN FIGURE  BEFORE CORRECTION #########
    plt.ion()
    fig = plt.figure(figsize=(8.3,3))
    # Page width is 8.3 inches
    # Create two plots: Picture and horizontal profiles
    ax1 = fig.add_subplot(121)
    ax2 = fig.add_subplot(143)
    ax3 = fig.add_subplot(144)
 
    # Given that the two figures on the right are very
    # close, we set the labels on the rightmost one
    # to the right
    ax3.yaxis.set_ticks_position('right')
     
    # If the limits of the y axis are not set, the first point will determine
    # the span of it
    ax2.set_ylim((0,255))
    ax3.set_ylim((0,255))
     
    # Prepare the cross to be updated later
    vlin = ax1.axvline(x=dataBefore.shape[1]/2)
    hlin = ax1.axhline(y=dataBefore.shape[0]/2)

    #Prepare the plots to be updated later. The generic first point is the central point in the image
     
    imag = ax1.imshow(dataBefore,cmap='gray') #norm=colors.Normalize(vmin=0,vmax=254,clip=True),origin='lower')
    row, = ax2.plot(dataBefore[dataBefore.shape[1]//2,:])
    column, = ax3.plot(dataBefore[:,dataBefore.shape[0]//2])
    # Format the axis
    ax1.set_yticks([dataBefore.shape[0]//2])
    ax1.set_xticks([dataBefore.shape[1]//2])
    ax1.set_title('Line Profile BEFORE Bias Field Correction')
     
    ax2.set_yticks([0, max(dataBefore[dataBefore.shape[0]//2,:]) ])
    ax2.set_title('Horizontal')
    ax2.set_xlim((0,dataBefore.shape[1]//2))
    ax2.set_xticks([])
     
    ax3.set_yticks([0, max(dataBefore[:,dataBefore.shape[0]//2]) ])
    ax3.set_title('Vertical')
    ax3.set_xlim((0,dataBefore.shape[1]//2))
    ax3.set_xticks([])

########## BEGIN FIGURE  AFTER CORRECTION #########
    plt.ion()
    fig_after = plt.figure(figsize=(8.3,3))
    # Page width is 8.3 inches
    # Create two plots: Picture and horizontal profiles
    ax4= fig_after.add_subplot(121)
    ax5= fig_after.add_subplot(143)
    ax6= fig_after.add_subplot(144)
    
     
    # Given that the two figures on the right are very
    # close, we set the labels on the rightmost one
    # to the right
    ax6.yaxis.set_ticks_position('right')
     
    # If the limits of the y axis are not set, the first point will determine
    # the span of it
    ax5.set_ylim((0,255))
    ax6.set_ylim((0,255))
     
    # Prepare the cross to be updated later
    vlin_after = ax4.axvline(x=dataAfter.shape[1]/2)
    hlin_after = ax4.axhline(y=dataAfter.shape[0]/2)

    #Prepare the plots to be updated later. The generic first point is the central point in the image
     
    imag_after = ax4.imshow(dataAfter,cmap='gray') 
    row_after, = ax5.plot(dataAfter[dataAfter.shape[1]//2,:])
    column_after,= ax6.plot(dataAfter[:,dataAfter.shape[0]//2])
    
   # Format the axis
    ax4.set_yticks([dataAfter.shape[0]//2])
    ax4.set_xticks([dataAfter.shape[1]//2])
    ax4.set_title('Line Profile AFTER Bias Field Correction')
     
    ax5.set_yticks([0, max(dataAfter[dataAfter.shape[0]//2,:]) ])
    ax5.set_title('Horizontal')
    ax5.set_xlim((0,dataAfter.shape[1]//2))
    ax5.set_xticks([])
     
    ax6.set_yticks([0, max(dataAfter[:,dataAfter.shape[0]//2]) ])
    ax6.set_title('Vertical')
    ax6.set_xlim((0,dataAfter.shape[1]//2))
    ax6.set_xticks([])
     

    fig3, ax7 = plt.subplots()
    # Plotting both the curves simultaneously
    ax7.plot(dataBefore[dataBefore.shape[1]//2,:],color='r', label='beforeCorrection')
    ax8 = ax7.twinx()
    ax8.plot(dataAfter[dataAfter.shape[1]//2,:], color='g', label='afterCorrection')

    # Naming the whole graph
    plt.title("Horizontal Line profile before vs after bias correction")

    # Adding legend, which helps us recognize the curve according to it's color
    plt.legend()

    # To load the display window
    plt.show()

    output_folder=r'DATA_ROOT/checking_results_v2' #Writethe path where you want to store your results
    final_fig=fig3.savefig(os.path.join(output_folder,mod,pat,pat+mod+'_LineProfiles.png'))
    plt.show()
    
    return final_fig

def ResamplingCheck(resampling_path,keyword):
    path2resample=resampling_path
    vol=nib.load(path2resample)
    info=vol.get_fdata()
    hdr=vol.header
    pixDim_all=hdr['pixdim']
    pixDim=(pixDim_all[1], pixDim_all[2], pixDim_all[3])
    matrixDim=np.array(vol.shape)
    print('Pixel dimension'+keyword+'resampling:')
    print(pixDim)
    print('Matrix dimension'+keyword+'resampling:')
    print(matrixDim)
    return pixDim,matrixDim


 

In [ ]:
#Functions PREPROCESSING

#1-PreProcess
def preProcess(path,pat_number,mod):  
    output_file=r'DATA_ROOT/preprocesado_results_v2'
    #1. Reorientation to MNI
    reo_to_std = Reorient2Std()  # Instantiation
    reo_to_std.inputs.in_file =path
    name_reo= pat_number.split('.')
    filename_reo= name_reo[0].split('/')
    print(filename_reo[-1])
    out_name=pat_number+mod+'_reoriented.nii.gz'
    out_file_path=os.path.join(output_file,mod,pat_number,out_name)
    reo_to_std.inputs.out_file =out_file_path
    res_reo_to_std = reo_to_std.run()
    #img=nib.load(res_reo_to_std.outputs.out_file)
    #nib.save(img,out_file_path)

    #!fslhd $res_reo_to_std.outputs.out_file  #Uncomment for visualization
    

     #2. Cropping of neck and shoulders
    robust_fov = RobustFOV()
    robust_fov.inputs.in_file = res_reo_to_std.outputs.out_file
    #robust_fov.inputs.in_file=os.path.join(output_file,mod,pat)
    robust_fov.inputs.brainsize=200 #Default is 170 but using larger values ensures the brain is not cropped.
    out_name_crop=pat_number+mod+'_cropped.nii.gz'
    out_file_path=os.path.join(output_file,mod,pat_number,out_name_crop)
    robust_fov.inputs.out_roi=out_file_path
    res_robust_fov = robust_fov.run()
    #img_crop=nib.load(res_robust_fov.outputs.out_roi)
    #nib.save(img_crop,out_file_path)

    #!fsleyes $res_robust_fov.outputs.out_roi

        # 3. Bias Field correction
    n4 = N4BiasFieldCorrection()
    n4.inputs.dimension = 3
    n4.inputs.input_image = res_robust_fov.outputs.out_roi
    #n4.inputs.output_image = os.path.join(output_file,mod,pat)
    n4.inputs.bspline_fitting_distance = 300
    n4.inputs.shrink_factor = 3
    n4.inputs.n_iterations = [50,50,30,20]
    n4.inputs.bias_image=os.path.join(output_file,mod,pat_number,pat_number+mod+'_biasField.nii.gz')
    out_name_corrected=pat_number+mod+'_biasCorrected.nii.gz'
    out_file_path=os.path.join(output_file,mod,pat_number,out_name_corrected)
    n4.inputs.output_image= out_file_path
    n4_out=n4.run()
    #img_bias=nib.load(n4_out.outputs.output_image)
    #nib.save(img_bias,out_file_path)
    return res_reo_to_std.outputs.out_file, res_robust_fov.outputs.out_roi,n4_out.outputs.output_image, n4_out.outputs.bias_image

#2-Resampling
#OPTION 1
def Resampling(path,pat,mod):
    nif_vol=nib.load(path)
    vol=nif_vol.get_fdata()
    res=resampleToPixdims(vol,[1,1,1])
    nif_res=nib.Nifti1Image(res,nif_vol.affine)
    path=os.path.join(output_file,mod,pat,pat+mod+'resampled.nii.gz')
    nib.save(nif_res,path)
    return path
#OPTION 2
def Resample(path,pat_number,mod):
    output_file=r'DATA_ROOT/preprocesado_results_v2'
    #output_file=r'DATA_ROOT/check_2'
    reslice = dipy.Resample()
    reslice.inputs.in_file = path
    reslice.inputs.interp=2
    reslice.inputs.vox_size=(0.718750,0.718750,0.718750)
    reslice_out=reslice.run() # doctest: +SKIP
    #!fsleyes $reslice_t1w_ce_out.outputs.out_file
    out_name_resample=pat_number+mod+'_resampled.nii.gz'
    out_file_path=os.path.join(output_file,mod,pat_number,out_name_resample)
    img_resample=nib.load(reslice_out.outputs.out_file)
    nib.save(img_resample,out_file_path)
    return reslice_out.outputs.out_file
  
    
 #3-BrainExtraction
def Brain_extraction(path,pat_number,mod): 
    output_file=r'DATA_ROOT/preprocesado_results_v2'
    #output_file=r'DATA_ROOT/check_2'
    brainextract = BrainExtraction()
    brainextract.inputs.dimension = 3
    brainextract.inputs.anatomical_image =path
    brainextract.inputs.brain_template = 'DATA_ROOT/Oasis/T_template0.nii.gz'
    brainextract.inputs.brain_probability_mask ='DATA_ROOT/Oasis/T_template0_BrainCerebellumProbabilityMask.nii.gz'
    brainextract.inputs.extraction_registration_mask='DATA_ROOT/Oasis/T_template0_BrainCerebellumRegistrationMask.nii.gz'
    brainextract.inputs.out_prefix=pat_number+mod
    extractedBrain=brainextract.run()
    out_name_brain=pat_number+mod+'_brainExtraction.nii.gz'
    out_name_mask=pat_number+mod+'_brainMask.nii.gz'
    out_file_path=os.path.join(output_file,mod,pat_number,out_name_brain)
    mask_path=os.path.join(output_file,mod,pat_number,out_name_mask)
    img_brainExtraction=nib.load( extractedBrain.outputs.BrainExtractionBrain)
    img_Mask=nib.load(extractedBrain.outputs.BrainExtractionMask)
    nib.save(img_brainExtraction,out_file_path)
    nib.save(img_Mask,mask_path)

    return extractedBrain.outputs.BrainExtractionBrain

def Registration(path_template,path_moving,pat_number,mod):
    output_file=r'DATA_ROOT/preprocesado_results_v2'
    #output_file=r'DATA_ROOT/check_2'
    lin_reg = FLIRT()
    lin_reg.inputs.in_file = path_moving
    #lin_reg.inputs.out_file = os.path.join(output_file,mod,pat)                   
    lin_reg.inputs.reference = path_template
    lin_reg.inputs.cost = "normmi"  # Same modality but not necessarily the same contrast 
    lin_reg.inputs.interp = "sinc"  # Sinc interpolation can be further configured if desired
    lin_reg.inputs.dof = 12  # Full affine transformation
    out_name_registration=pat_number+mod+'_registered.nii.gz'
    out_file_path=os.path.join(output_file,mod,pat_number,out_name_registration)
    lin_reg.inputs.out_file=out_file_path
    res_lin_reg = lin_reg.run()
    matrix=res_lin_reg.outputs.out_matrix_file 
    
    #img_registered=nib.load(res_lin_reg.outputs.out_file)
    #nib.save(img_registered,out_file_path)

    return res_lin_reg.outputs.out_file, matrix

def MeasureSimilarity(path_template,path_moving):
  #Quantify similarity
    sim = MeasureImageSimilarity()
    sim.inputs.dimension = 3
    sim.inputs.metric = 'MI'
    sim.inputs.fixed_image = path_template
    sim.inputs.moving_image = path_moving
    sim.inputs.metric_weight = 1.0
    sim.inputs.radius_or_number_of_bins = 50 #Play with this parameter
    sim.inputs.sampling_strategy = 'Regular'
    sim.inputs.sampling_percentage = 1
    sim_out=sim.run()
    return sim_out.outputs.similarity


#For inserting images:
# def nifti2jpg(nii_folder,nii_slice_folder,jpg_folder):
#     nii_img=nib.load(nii_folder)
#     info_nii=nii_img.get_fdata()
#     one_slice=info_nii[:,:,info_nii.shape[0]//4]
#     one_slice_nii=nib.Nifti1Image(one_slice, nii_img.affine, nii_img.header)
#     nib.save(one_slice_nii,nii_slice_folder)
#     jpg_img=nib.load(nii_slice_folder).get_fdata()/8
#     cv2.imwrite(jpg_folder,jpg_img)
#     return jpg_folder



In [ ]:
import xlsxwriter
mod_list=['T1W_CE_3D_FFE_TRA_IQ','T1W_3D_FFE_TRA_IQ', 'FLAIR_3D_SPIR_TRA_IQ', 'SWIm']
template='T1W_CE_3D_FFE_TRA_IQ'
content_row = ['PAT_'+ str(pat+1) for pat in range(3)]

#Load into nibabel
#input_path=r'DATA_ROOT/original_data'
input_path='/media/sf_prueba_datos_originales'
output_file=r'DATA_ROOT/preprocesado_results_v2'
root_path=r'DATA_ROOT/checking_results_v2'
#brain_path=r'/media/sf_resultados_preprocesado/checking_procesado'
#meterse en cada carpeta (que es una modalidad distinta)
orientation_list=[]
pixel_size_before_list=[]
pixel_size_after_size=[]
matrix_size_before_list=[]
matrix_size_after_list=[]

#START EXCEL
workbook = xlsxwriter.Workbook(os.path.join(root_path,'PreProcessingCheck2.xlsx'))

#Sheet1: Original
worksheet = workbook.add_worksheet(name='Original')
worksheet.set_default_row(50)
row = 2
column = 0
# Widen the first column to make the text clearer.
for col in range(11):
    worksheet.set_column(col,col, 70)

wrap_format = workbook.add_format({'text_wrap': True})
bold = workbook.add_format({'bold': True})
worksheet.write('A1', 'PreProcessing Check')
worksheet.write('A2', 'PAT_NUM', bold)
worksheet.write('B2', 'T1-CE', bold)
worksheet.write('C2', 'T1', bold)
worksheet.write('D2', 'FLAIR', bold)
worksheet.write('E2', 'swim', bold)

for item in content_row:
    worksheet.write(row, column, item)
    row+=3

#Sheet2: Reorientation
worksheet = workbook.add_worksheet(name='Reorientation')
worksheet.set_default_row(50)
row = 2
column = 0
for col in range(11):
    worksheet.set_column(col,col, 30)


worksheet.write('A1', 'PreProcessing Check')
worksheet.write('B1','REORIENTATION')
worksheet.write('A2', 'PAT_NUM', bold)
worksheet.write('B2', 'T1-CE', bold)
worksheet.write('C2', 'T1', bold)
worksheet.write('D2', 'FLAIR', bold)
worksheet.write('E2', 'swim', bold)

for item in content_row:
    worksheet.write(row, column, item)
    row+=3

#Sheet3: Bias Field
worksheet = workbook.add_worksheet(name='Bias Field') 
worksheet.set_default_row(70) 
row = 2
column = 0
for col in range(11):
    worksheet.set_column(row,col, 70)


worksheet.write('A1', 'PreProcessing Check')
worksheet.write('B1','BIAS FIELD CORRECTION')
worksheet.write('A2', 'PAT_NUM', bold)
worksheet.write('B2', 'T1-CE', bold)
worksheet.write('C2', 'T1', bold)
worksheet.write('D2', 'FLAIR', bold)
worksheet.write('E2', 'swim', bold)

for item in content_row:
    worksheet.write(row, column, item)
    row+=3

#Sheet4:RESAMPLING
worksheet = workbook.add_worksheet(name='Resampling') 
worksheet.set_default_row(70) 
row = 2
column = 0
for col in range(11):
    worksheet.set_column(row,col, 70)


worksheet.write('A1', 'PreProcessing Check')
worksheet.write('B1','RESAMPLING')
worksheet.write('A2', 'PAT_NUM', bold)
worksheet.write('B2', 'T1-CE', bold)
worksheet.write('C2', 'T1', bold)
worksheet.write('D2', 'FLAIR', bold)
worksheet.write('E2', 'swim', bold)


for item in content_row:
    worksheet.write(row, column, item)
    row+=5

#Sheet5:BrainExtraction

worksheet = workbook.add_worksheet(name='BrainExtraction') 
worksheet.set_default_row(50) 
row = 2
column = 0
for col in range(11):
    worksheet.set_column(row,col, 30)


worksheet.write('A1', 'PreProcessing Check', bold)
worksheet.write('B1','BRAIN EXTRACTION', bold)
worksheet.write('A2','PAT_NUM', bold)
worksheet.write('B2', 'T1-CE', bold)
worksheet.write('C2', 'T1', bold)
worksheet.write('D2', 'FLAIR', bold)
worksheet.write('E2', 'swim', bold)


for item in content_row:
    worksheet.write(row, column, item)
    row+=2

#Sheet6:REGISTRATION

worksheet = workbook.add_worksheet(name='Registration') 
worksheet.set_default_row(50) 
row = 2
column = 0
for col in range(11):
    worksheet.set_column(col,col, 30)


worksheet.write('A1', 'PreProcessing Check')
worksheet.write('B1','REGISTRATION')
worksheet.write('A2','PAT_NUM', bold)
worksheet.write('B2', 'T1-CE', bold)
worksheet.write('C2', 'T1', bold)
worksheet.write('D2', 'FLAIR', bold)
worksheet.write('E2', 'swim', bold)

for item in content_row:
    worksheet.write(row, column, item)
    row+=3
    

# Start from the first cell.
# Rows and columns are zero indexed.
row_reorient_temp=2
row_reorient_mod=2
row_bias_temp = 2
row_bias_mod=2
row_resamp_temp=2
row_resamp_mod=2
row_brain_temp=2
row_brain_mod=2
row_register=2

column_reorient_temp=0
column_reorient_mod=1
column_bias_temp = 0
column_bias_mod=1
column_resamp_temp=0
column_resamp_mod=1
column_brain_temp=0
column_brain_mod=1
column_register=1


image_width = 182.0
image_height = 182.0

cell_width = 64.0
cell_height = 64.0

x_scale = cell_width/image_width
y_scale = cell_height/image_height




for pat in range(3,4):
    for mod in mod_list:
        print(mod)
        if mod==template: 
            pat_number='PAT_'+str(pat)+'_'
            print(pat_number)
            os.makedirs(os.path.join(output_file,mod,pat_number))
            os.makedirs(os.path.join(root_path,mod,pat_number))

            path_t1w_ce=os.path.join(input_path,mod,pat_number+template+'.nii')
            
            #1-PreProcessing template
            preProcess_t1w_ce=preProcess(path_t1w_ce,pat_number,mod)

            #Check REPROCESSING
            #1.1-Reorientation
            path_t1w_ce_reoriented=os.path.join(output_file,template,pat_number,pat_number+ template+'_reoriented.nii.gz')
            t1w_ce_orientationCheck=OrientationCheck(path_t1w_ce_reoriented)
            print(t1w_ce_orientationCheck)
            #Insert image
            t1_ce_reoriented_jpg=nifti2jpg(path_t1w_ce_reoriented, pat_number, template, 'reoriented')
            t1_ce_reoriented_read=plt.imread(t1_ce_reoriented_jpg)
            plt.imshow(t1_ce_reoriented_read, cmap='gray')
            plt.axis('off')
            plt.show()
            #Excel sheet-REORIENTATION temp
            worksheet=workbook.get_worksheet_by_name('Reorientation')
            column_reorient_temp+=1
            worksheet.write(row_reorient_temp,column_reorient_temp,'Orientation:'+' '+ t1w_ce_orientationCheck,wrap_format )
            row_reorient_temp+=1
            worksheet.insert_image(row_reorient_temp,column_reorient_temp,os.path.join(root_path,pat_number,pat_number+template+'_reoriented_slice.jpg'),{'x_scale': x_scale, 'y_scale': y_scale} )
           
            
            #1.2-Crop Neck and Shoulders
            cropNeck_path_temp= os.path.join(output_file,template,pat_number,pat_number+template+'_cropped.nii.gz') #preProcess_t1w_ce.res_robust_fov.outputs.out_roi
            cropNeck_temp_jpg=nifti2jpg(cropNeck_path_temp, pat_number, template, 'cropped')
            #1.3-Bias Field Correction
            NoBias_path_temp= os.path.join(output_file, template,pat_number,pat_number+template+'_biasCorrected.nii.gz') #preProcess_t1w_ce.n4_out.outputs.output_image
            NoBias_temp_jpg=nifti2jpg(NoBias_path_temp, pat_number, template, 'biasCorrected')
            #Line Profile 
            Bias_jpg_temp=BiasFieldCheck(cropNeck_temp_jpg,NoBias_temp_jpg,pat_number,template)
            

            #Plot bias field
            bias_field_path=os.path.join(output_file, template,pat_number,pat_number+template+'_biasField.nii.gz')
            bias_field_jpg=nifti2jpg(bias_field_path, pat_number, template, 'biasField')
            image = mpimg.imread(bias_field_jpg)
            plt.imshow(image)
            plt.show()
            
            #Excel-Bias Field-TEMP
            worksheet=workbook.get_worksheet_by_name('Bias Field')
            column_bias_temp+=1
            worksheet.insert_image(row_bias_temp,column_bias_temp, os.path.join(root_path,pat_number,pat_number+template+'_LineProfiles.png'), {'x_scale': x_scale, 'y_scale': y_scale})
            row_bias_temp+=2
            worksheet.insert_image(row_bias_temp,column_bias_temp, os.path.join(root_path,pat_number,pat_number+template+'_biasCorrected_slice.jpg'), {'x_scale': x_scale, 'y_scale': y_scale})
            row_bias_temp+=2
            worksheet.insert_image(row_bias_temp,column_bias_temp, os.path.join(root_path,pat_number,pat_number+template+'_biasField_slice.jpg'), {'x_scale': x_scale, 'y_scale': y_scale})
            
            

            #2-Resampling template
            Resampling_t1w_ce=Resample(preProcess_t1w_ce[2],pat_number,mod)

            # Checking RESAMPLING
            #Pixel and matrix size before resampling
            #data_BeforeResampling_temp=NoBias_path_temp 
            #BeforeResampling_temp_jpg= NoBias_temp_jpg
            BeforeResampling_temp=ResamplingCheck(preProcess_t1w_ce[2],' before') #path to volume
            BeforeResampling_pixels=pixel_size_before_list.append(BeforeResampling_temp[0])
            BeforeResampling_matrix=matrix_size_before_list.append(BeforeResampling_temp[1])

            #Pixel and matrix size after resampling
            data_AfterResampling_temp=os.path.join(output_file,template,pat_number,pat_number+template+'_resampled.nii.gz')
            AfterResampling_temp_jpg=nifti2jpg(data_AfterResampling_temp,pat_number,template,'resampled')
            AfterResampling_temp=ResamplingCheck(data_AfterResampling_temp,' after ') #path to volume
            AfterResampling_pixels=pixel_size_after_size.append(AfterResampling_temp[0])
            AfterResampling_matrix=matrix_size_after_list.append(AfterResampling_temp[1])
            
            #Excel-Resampling TEMP
            
            worksheet=workbook.get_worksheet_by_name('Resampling')
            column_resamp_temp+=1
            #Pixel and matrix size before resampling
            worksheet.write(row_resamp_temp,column_resamp_temp,'Pixel size before resampling:'+' '+ str(BeforeResampling_temp[0]), wrap_format)
            row_resamp_temp += 1
            worksheet.write(row_resamp_temp,column_resamp_temp,'Matrix size before resampling:'+' ' + str(BeforeResampling_temp[1]), wrap_format)
            row_resamp_temp+=1

            #Pixel size after resampling
            worksheet.write(row_resamp_temp,column_resamp_temp,'Pixel size after resampling:'+' '+str(AfterResampling_temp[0]), wrap_format)
            row_resamp_temp += 1
            worksheet.write(row_resamp_temp,column_resamp_temp,'Matrix size after resampling:'+' '+str(AfterResampling_temp[1]), wrap_format)
            row_resamp_temp += 1
            # Insert an image.
            worksheet.insert_image(row_resamp_temp,column_resamp_temp,os.path.join(root_path,pat_number,pat_number+mod+'_resampled_slice.jpg') , {'x_scale': x_scale, 'y_scale': y_scale})
            
            
            

            #3-BrainExtraction template
            BrainExtract_t1w_ce_brain=Brain_extraction(Resampling_t1w_ce,pat_number,mod)
            #!fsleyes $BrainExtract_t1w_ce
            
            #Excel-Brain Extraction TEMP
            #worksheet=workbook.get_worksheet_by_name('Brain Extraction')
            #column_brain_temp+=1
            #worksheet.insert_image(row_brain_mod,column_brain_mod,os.path.join(brain_path,mod,pat_number,pat_number+mod+'_brainExtraction.png'), {'x_scale': x_scale, 'y_scale': y_scale})
            
            
        else:
            os.makedirs(os.path.join(output_file,mod,pat_number))
            os.makedirs(os.path.join(root_path,mod,pat_number))
            path_mod=os.path.join(input_path,mod,pat_number+mod+'.nii')

            #1-PreProcessing modality
            preProcess_mod=preProcess(path_mod,pat_number,mod)

            #Checking PreProcessing
            #1.1-Orientation
            path_mod_reoriented=os.path.join(output_file,mod,pat_number,pat_number+mod+'_reoriented.nii.gz')
            mod_orientationCheck=OrientationCheck (path_mod_reoriented)
            mod_orientation_values=orientation_list.append(mod_orientationCheck)
            print(mod_orientationCheck)
            #Insert image
            mod_reoriented_jpg=nifti2jpg(path_mod_reoriented,pat_number,mod, 'reoriented')
            img_reoriented=plt.imread(mod_reoriented_jpg)
            plt.imshow(img_reoriented, cmap='gray')
            plt.axis('off')
            plt.show()

            #Excel-Reorientation mod
            worksheet=workbook.get_worksheet_by_name('Reorientation')
            column_reorient_mod+=1
            worksheet.write(row_reorient_mod,column_reorient_mod,'Orientation:'+' '+ mod_orientationCheck, wrap_format)
            row_reorient_mod+=1
            # Insert an image.
            worksheet.insert_image(row_reorient_mod,column_reorient_mod,os.path.join(root_path,pat_number,pat_number+mod+'_reoriented_slice.jpg'), {'x_scale': x_scale, 'y_scale':y_scale})
            
                
                
            #1.2-Crop Neck and Shoulders
            cropNeck_path=os.path.join(output_file,mod,pat_number,pat_number+mod+'_cropped.nii.gz')
            cropNeck_jpg=nifti2jpg(cropNeck_path, pat_number,mod, 'cropped')
            #1.3-Bias Field Correction
            NoBias_path= os.path.join(output_file,mod,pat_number,pat_number+mod+'_biasCorrected.nii.gz') #preProcess_mod.n4_out.outputs.output_image
            NoBias_jpg=nifti2jpg(NoBias_path, pat_number, mod, 'biasCorrected')
            #Line Profile 
            Bias_jpg=BiasFieldCheck(cropNeck_jpg,NoBias_jpg,pat_number,mod)


            #Plot bias field
            bias_field_path_mod=os.path.join(output_file,mod,pat_number,pat_number+mod+'_biasField.nii.gz')
            bias_field_mod_jpg=nifti2jpg(bias_field_path_mod, pat_number,mod, 'biasField')
            image = mpimg.imread(bias_field_mod_jpg)
            plt.imshow(image)
            plt.show()
            
            #Excel-BIAS FIELD-MOD
                                   
            worksheet=workbook.get_worksheet_by_name('Bias Field')
            column_bias_mod+=1
            worksheet.insert_image(row_bias_mod,column_bias_mod, os.path.join(root_path,pat_number,pat_number+mod+'_LineProfiles.png'), {'x_scale': x_scale, 'y_scale': y_scale})
            row_bias_mod+=2
            worksheet.insert_image(row_bias_mod,column_bias_mod, os.path.join(root_path,pat_number,pat_number+mod+'_biasCorrected_slice.jpg'), {'x_scale': x_scale, 'y_scale': y_scale})
            row_bias_mod+=2
            worksheet.insert_image(row_bias_mod,column_bias_mod, os.path.join(root_path,pat_number,pat_number+mod+'_biasField_slice.jpg'), {'x_scale': x_scale, 'y_scale': y_scale})
            


            #2-Resampling modality
            Resampling_mod=Resample(preProcess_mod[2],pat_number,mod)

            #2-Checking RESAMPLING
            #Pixel and matrix size before resampling
            #data_BeforeResampling=NoBias_path 
            BeforeResampling=ResamplingCheck(preProcess_mod[2],'before')
            BeforeResampling_pixels=pixel_size_before_list.append(BeforeResampling[0])
            BeforeResampling_matrix=matrix_size_before_list.append(BeforeResampling[1])

            #Image before resampling
            NoResampling_img=NoBias_jpg

            #Pixel and matrix size after resampling
            data_AfterResampling=os.path.join(output_file,mod,pat_number,pat_number+mod+'_resampled.nii.gz')
            AfterResampling=ResamplingCheck(data_AfterResampling,'after') #Path to the volume
            AfterResampling_pixels=pixel_size_after_size.append(AfterResampling[0])
            AfterResampling_matrix=matrix_size_after_list.append(AfterResampling[1])
            #Image after resampling
            Resampling_img=nifti2jpg(data_AfterResampling,pat_number,mod,'resampled')
            
            #Excel-Resampling MOD
            
            worksheet=workbook.get_worksheet_by_name('Resampling')
            column_resamp_mod+=1
            #Pixel and matrix size before resampling
            worksheet.write(row_resamp_mod,column_resamp_mod,'Pixel size before resampling:'+' '+ str(BeforeResampling[0]), wrap_format)
            row_resamp_mod += 1
            worksheet.write(row_resamp_mod,column_resamp_mod,'Matrix size before resampling:'+' ' + str(BeforeResampling[1]), wrap_format)
            row_resamp_mod+=1

            #Pixel size after resampling
            worksheet.write(row_resamp_mod,column_resamp_mod,'Pixel size after resampling:'+' '+str(AfterResampling[0]), wrap_format)
            row_resamp_mod += 1
            worksheet.write(row_resamp_mod,column_resamp_mod,'Matrix size after resampling:'+' '+str(AfterResampling[1]), wrap_format)
            row_resamp_mod += 1
            # Insert an image.
            worksheet.insert_image(row_resamp_mod,column_resamp_mod,os.path.join(root_path,pat_number,pat_number+mod+'_resampled_slice.jpg') , {'x_scale': x_scale, 'y_scale': y_scale})
            
            

            #3-BrainExtraction modality
            BrainExtract_mod_brain =Brain_extraction(Resampling_mod,pat_number,mod)
            #!fsleyes $BrainExtract_mod

            #Excel-Brain Extraction-MOD
            #worksheet=workbook.get_worksheet_by_name('Brain Extraction')
            #column_brain_mod+=1
            #worksheet.insert_image(row_brain_mod,column_brain_mod,os.path.join(brain_path,mod,pat_number,pat_number+mod+'_brainExtraction.png'), {'x_scale': x_scale, 'y_scale': y_scale})
            


            #4-REGISTRATION
            #Similarity before registration
            similarity_before=MeasureSimilarity(BrainExtract_t1w_ce_brain,BrainExtract_mod_brain)
            print('Similarity before registration:',similarity_before)

            #Registration
            register_mod, matrix=Registration(BrainExtract_t1w_ce_brain,BrainExtract_mod_brain,pat_number,mod)
            nii_Registered=os.path.join(output_file,mod,pat_number,pat_number+mod+'_registered.nii.gz')
            jpg_Registration=nifti2jpg(nii_Registered,pat_number,mod,'registered')

            #Similarity after registration
            similarity_after=MeasureSimilarity(BrainExtract_t1w_ce_brain,register_mod)

            print('Similarity after registration:',similarity_after)

            #Excel-registration
            worksheet=workbook.get_worksheet_by_name('Registration')
            column_register+=1
             #similarity before registration
            worksheet.write(row_register,column_register,'Image similarity before registration:'+' '+str(similarity_before))
            row_register+=1

            #Image similarity after registration
            worksheet.write(row_register,column_register,'Image similarity after registration:'+' '+str(similarity_after))
            row_register+=1
            
            #Matrix
            worksheet.write(row_register,column_register,'Similarity matrix:'+' '+matrix)
            
            
        
        row_register-=2
        row_resamp_mod-=4
        row_reorient_mod-=1
        row_bias_mod-=5
                                   
            
        


    column_reorient_temp=0
    column_reorient_mod=1
    column_bias_temp = 0
    column_bias_mod=1
    column_resamp_temp=0
    column_resamp_mod=1
    column_brain_temp=0
    column_brain_mod=1
    column_register=1
    row_brain_temp+=1                               
    row_brain_mod+=2
    row_resamp_temp+=1
    row_resamp_mod+=5
    row_reorient_temp+=1                  
    row_reorient_mod+=2
    row_bias_temp+=1
    row_bias_mod+=6
    row_register+=3
        
        
workbook.close()



In [ ]:
fsl.FLIRT().inputs_help()


In [ ]:
#REGISTRATION
similarity_list_before=[]
similarity_list_after=[]
for mod in os.listdir(input_path):
    modality_path=os.path.join(input_path,mod)
    print(mod)
    if mod==template:
        print('No registration')
    else:
        for pat in os.listdir(modality_path):
            #Similarity before registration
            similarity_before=MeasureSimilarity(BrainExtract_t1w_ce,BrainExtract_mod)
            similarity_list_before.append(similarity_before)
            print('Similarity before registration:',similarity_before)
            print('Similarity before registration(2):', similarity_list_before[pat])
            
            #4-Registration
            register_mod=Registration(BrainExtract_t1w_ce,BrainExtract_mod,pat_number,mod)
            nii_Registered=os.path.join(root_path,mod,pat_number,pat_number+'_'+mod+'_registered.nii.gz')
            jpg_Registration=nifti2jpg(nii_Registered,pat_number,mod,'registered')
            
            #Similarity after registration
            similarity_after=MeasureSimilarity(BrainExtract_t1w_ce,register_mod)
            similarity_list_after.append(similarity_after)
            print('Similarity after registration:',similarity_after)
            print('SIMILARITY AFTER REGISTRATION:', similarity_list_after[pat])

            #Insert image
            Registration_img=plt.imread(jpg_Registration)
            plt.imshow(Registration_img, cmap='gray')
            plt.axis('off')
            plt.show()
        

In [ ]:

##Excel file
import xlsxwriter
import os



def scale(size):
    row=100
    col=50
    return (row/size[0],col/size[1])

x,y=scale([420,250])

output_folder=r'/media/sf_resultados_preprocesado/checking_procesado'
#original_folder=r'/media/sf_prueba_datos_originales/'


workbook = xlsxwriter.Workbook(output_folder+'\\'+'PreProcessingCheck.xlsx')

#Sheet1: Original
worksheet = workbook.add_worksheet(name='Original')
worksheet.set_default_row(50)
# Widen the first column to make the text clearer.
for col in range(120):
    worksheet.set_column(col,col, 30)

wrap_format = workbook.add_format({'text_wrap': True})
bold = workbook.add_format({'bold': True})
worksheet.write('A1', 'PreProcessing Check')
worksheet.write('A2', 'PAT_NUM', bold)
worksheet.write('B2', 'T1-CE', bold)
worksheet.write('C2', 'T1', bold)
worksheet.write('D2', 'FLAIR', bold)
worksheet.write('E2', 'swim', bold)

# Start from the first cell.
# Rows and columns are zero indexed.
row = 2
column = 0


content_row = ['PAT_'+ str(pat+1) for pat in range(120)]
content_column=['T1W_3D_FFE_TRA_IQ', 'FLAIR_3D_SPIR_TRA_IQ', 'SWIm']
template='T1W_CE_3D_FFE_TRA_IQ'


 
# iterating through content list
# for item in content_row:
#     # write number of the patient in the first column
#     worksheet.write(row, column, item)
#     column+=1
#     worksheet.insert_image(row,column, original_folder+'\\'+item+'\\'+item+'_T1W_CE_3D_FFE_TRA_IQ_original_slice.jpg',  {'x_scale': 2*x, 'y_scale': 2*y})
#     column+=1
#     for mod in content_column:
#         # Insert an image.
#         worksheet.insert_image(row,column, original_folder+'\\'+item+'\\'+item+'_'+mod+'_original_slice.jpg', {'x_scale': 2*x, 'y_scale': 2*y})
#         column+=1
#     column=0
#     row+=2
        
#Sheet2: Reorientation
worksheet = workbook.add_worksheet(name='Reorientation')
worksheet.set_default_row(50)
for col in range(120):
    worksheet.set_column(col,col, 30)


worksheet.write('A1', 'PreProcessing Check')
worksheet.write('B1','REORIENTATION')
worksheet.write('A2', 'PAT_NUM', bold)
worksheet.write('B2', 'T1-CE', bold)
worksheet.write('C2', 'T1', bold)
worksheet.write('D2', 'FLAIR', bold)
worksheet.write('E2', 'swim', bold)
row = 2
column = 0
# iterating through content list
for item in content_row:
    # write operation perform
    worksheet.write(row, column, item)
    column+=1
    worksheet.write(row,column,'Orientation:'+' '+ t1w_ce_orientationCheck,wrap_format )
    row+=1
    worksheet.insert_image(row,column,os.path.join(output_folder,item,item+'_'+template+'_reoriented_slice.jpg'),{'x_scale': 2*x, 'y_scale': 2*y} )
    column+=1
    for idx, mod in enumerate(content_column): 
        row-=1
        worksheet.write(row,column,'Orientation:'+' '+ orientation_list[idx], wrap_format)
        row+=1
        # Insert an image.
        worksheet.insert_image(row,column, output_folder+'\\'+item+'\\'+item + '_'+ mod+'_reoriented_slice.jpg', {'x_scale': 2*x, 'y_scale': 2*y})
        column+=1
    row+=2
    column=0

    
    
#Sheet3: Bias Field
worksheet = workbook.add_worksheet(name='Bias Field') 
worksheet.set_default_row(50) 
row=2
for col in range(120):
    worksheet.set_column(row,col, 50)


worksheet.write('A1', 'PreProcessing Check')
worksheet.write('B1','BIAS FIELD CORRECTION')
worksheet.write('A2', 'PAT_NUM', bold)
worksheet.write('B2', 'T1-CE', bold)
worksheet.write('C2', 'T1', bold)
worksheet.write('D2', 'FLAIR', bold)
worksheet.write('E2', 'swim', bold)
row = 2
column = 0
# iterating through content list
for item in content_row:
    # write operation perform
    worksheet.write(row, column, item)
    column+=1
    #Template
    #Insert an image.
    worksheet.insert_image(row,column, os.path.join(output_folder,item,item+'_'+template+'_LineProfiles.png'), {'x_scale': 0.4, 'y_scale': 0.4})
    row+=2
    worksheet.insert_image(row,column, os.path.join(output_folder,item,item+'_'+template+'_biasCorrected_slice.jpg'), {'x_scale': 0.4, 'y_scale': 0.4})
    row+=2
    worksheet.insert_image(row,column, os.path.join(output_folder,item,item+'_'+template+'_biasField_slice.jpg'), {'x_scale': 0.4, 'y_scale': 0.4})
    column+=1
    for mod in content_column: 
        # Insert an image.
        row-=4
        worksheet.insert_image(row,column,os.path.join(output_folder,item,item+'_'+mod+'_LineProfiles.png'), {'x_scale': 0.4, 'y_scale': 0.4})
        row+=2
    worksheet.insert_image(row,column, os.path.join(output_folder,item,item+'_'+mod+'_biasCorrected_slice.jpg'), {'x_scale': 0.4, 'y_scale': 0.4})
    row+=2
    worksheet.insert_image(row,column, os.path.join(output_folder,item,item+'_'+mod+'_biasField_slice.jpg'), {'x_scale': 0.4, 'y_scale': 0.4})
        column+=1
    column=0
    row+=1

#Sheet4:RESAMPLING

worksheet = workbook.add_worksheet(name='Resampling') 
worksheet.set_default_row(50) 
row=2
for col in range(120):
    worksheet.set_column(row,col, 50)


worksheet.write('A1', 'PreProcessing Check')
worksheet.write('B1','RESAMPLING')
worksheet.write('A2', 'PAT_NUM', bold)
worksheet.write('B2', 'T1-CE', bold)
worksheet.write('C2', 'T1', bold)
worksheet.write('D2', 'FLAIR', bold)
worksheet.write('E2', 'swim', bold)
row = 2
column = 0
# iterating through content list
for item in content_row:
 
    # write operation perform
    worksheet.write(row, column, item)
    
    column += 1
    #Pixel and matrix size before resampling templat
    worksheet.write(row,column,'Pixel size before resampling:'+' '+ str(BeforeResampling_temp[0]), wrap_format)
    row+=1
    worksheet.write(row,column,'Matrix size before resampling:'+' '+ str(BeforeResampling_temp[1]), wrap_format)
    row+=1
    # Insert an image.
    worksheet.insert_image(row,column, os.path.join(output_folder,item,item+'_'+template+'_resampled_slice.jpg') , {'x_scale': 0.4, 'y_scale': 0.4})
    row+=2

    
    #Pixel size after resampling template
    worksheet.write(row,column,'Pixel size after resampling:'+' '+str(AfterResampling_temp[0]), wrap_format)
    row += 1
    worksheet.write(row,column,'Matrix size after resampling:'+' '+str(AfterResampling_temp[1]), wrap_format)
    row += 1
    # Insert an image.
    worksheet.insert_image(row,column,os.path.join(output_folder,item,item+'_'+template+'_resampled_slice.jpg') , {'x_scale': 0.4, 'y_scale': 0.4})
    column+=1

    for mod in content_column:
        row-=6
        #Pixel and matrix size before resampling
        worksheet.write(row,column,'Pixel size before resampling:'+' '+ str(BeforeResampling_pixels), wrap_format)
        row += 1
        worksheet.write(row,column,'Matrix size before resampling:'+' ' + str(BeforeResampling_matrix), wrap_format)
        row+=1
        # Insert an image.
        worksheet.insert_image(row,column, os.path.join(output_folder,item,item+'_'+mod+'_resampled_slice.jpg') , {'x_scale': 0.4, 'y_scale': 0.4})
        row+=2

    
        #Pixel size after resampling
        worksheet.write(row,column,'Pixel size after resampling:'+' '+str(AfterResampling_pixels), wrap_format)
        row += 1
        worksheet.write(row,column,'Matrix size after resampling:'+' '+str(AfterResampling_matrix), wrap_format)
        row += 1
        # Insert an image.
        worksheet.insert_image(row,column,os.path.join(output_folder,item,item+'_'+mod+'_resampled_slice.jpg') , {'x_scale': 0.4, 'y_scale': 0.4})
        column+=1

    column=0
    row+=2
    
#Sheet5:BrainExtraction

worksheet = workbook.add_worksheet(name='BrainExtraction') 
worksheet.set_default_row(50) 
row=2
for col in range(120):
    worksheet.set_column(row,col, 30)


worksheet.write('A1', 'PreProcessing Check', bold)
worksheet.write('B1','BRAIN EXTRACTION', bold)
worksheet.write('A2','PAT_NUM', bold)
worksheet.write('B2', 'T1-CE', bold)
worksheet.write('C2', 'T1', bold)
worksheet.write('D2', 'FLAIR', bold)
worksheet.write('E2', 'swim', bold)
row = 2
column = 0
# iterating through content list
for item in content_row:
 
    # write operation perform
    worksheet.write(row, column, item)
    column+=1
    worksheet.insert_image(row,column,os.path.join(output_folder,item,item+'_'+template+'_brainExtraction.png'), {'x_scale': 0.4, 'y_scale': 0.4})
    column+=1
    for mod in content_column:     
        worksheet.insert_image(row,column,os.path.join(output_folder,item,item+'_'+mod+'_brainExtraction.png'), {'x_scale': 0.4, 'y_scale': 0.4})
        column+=1
        
    column=0
    row+=2
    
#Sheet6:REGISTRATION

worksheet = workbook.add_worksheet(name='Registration') 
worksheet.set_default_row(50) 
for col in range(120):
    worksheet.set_column(col,col, 30)


worksheet.write('A1', 'PreProcessing Check')
worksheet.write('B1','REGISTRATION')
worksheet.write('A2','PAT_NUM', bold)
worksheet.write('B2', 'T1-CE', bold)
worksheet.write('C2', 'T1', bold)
worksheet.write('D2', 'FLAIR', bold)
worksheet.write('E2', 'swim', bold)
row = 2
column = 0
# iterating through content list
for item in content_row:
 
    # write operation perform
    worksheet.write(row, column, item)
    column+=2
    for mod in content_column:
        
        #similarity before registration
        worksheet.write(row,column,'Image similarity before registration:'+' '+similarity_before)
        row+=1
        #Insert image before registration
        worksheet.write(row,column,'Image before registration', wrap_format)
        row+=1
        worksheet.insert_image(row,column,os.path.join(output_folder,item,item+'_'+mod+'_brainExtraction.png'), {'x_scale': 0.4, 'y_scale': 0.4})
        row+=2
 
    
        #Image similarity after registration
        worksheet.write(row,column,'Image similarity after registration:'+' '+similarity_after)
        row+=1
        #Insert image after registration
        worksheet.write(row,column,'Image after registration', wrap_format)
        row+=1
        worksheet.insert_image(row,column,os.path.join(output_folder,item,item+'_'+mod+'_registered_slice.jpg'), {'x_scale': 0.4, 'y_scale': 0.4})
        column+=1
        row-=4

    column=0
    row+=6
workbook.close()

        




        
        




In [ ]:

import skimage.io as io
def nifti2jpg(nii_folder,pat_num,mod,step):
    root_path=r'DATA_ROOT\3_StructuralPreProcessing\resultados_preprocesado_prueba_novale\checking_results_v2' #Write the path where you want to store the jpg images
    nii_slice_folder=os.path.join(root_path,pat_num,pat_num+mod+'_' +step+'_slice'+'.nii.gz')
    jpg_folder=os.path.join(root_path,mod,pat_num,pat_num+ mod+'_' +step+'_slice.jpg')
    nii_img=nib.load(nii_folder)
    info_nii=nii_img.get_fdata()
    my_slice=info_nii[:,:,119]
    io.imsave(jpg_folder,my_slice)
    return jpg_folder

def BiasFieldCheck(pathLikeBefore,pathLikeAfter,pat,mod):
    filenameBefore=pathLikeBefore #Aqui va el nombre del archivo before bias field
    filenameAfter=pathLikeAfter

    #Read the figure before correction
    dataBefore=imread(filenameBefore)
    #Read the figure after correction
    dataAfter=imread(filenameAfter)


    #Obtain the position of the pixel with the maximum value BEFORE correction
    maxrows = np.argmax(dataBefore,axis=0)
    maxrowsVal = dataBefore
    
    #Obtain the position of the pixel with the maximum value AFTER correction
    maxrows_after = np.argmax(dataAfter,axis=0)
    maxrowsVal_after = dataAfter
    
    ########## BEGIN FIGURE  BEFORE CORRECTION #########
    plt.ion()
    fig = plt.figure(figsize=(8.3,3))
    # Page width is 8.3 inches
    # Create two plots: Picture and horizontal profiles
    ax1 = fig.add_subplot(121)
    ax2 = fig.add_subplot(143)
    ax3 = fig.add_subplot(144)
 
    # Given that the two figures on the right are very
    # close, we set the labels on the rightmost one
    # to the right
    ax3.yaxis.set_ticks_position('right')
     
    # If the limits of the y axis are not set, the first point will determine
    # the span of it
    ax2.set_ylim((0,255))
    ax3.set_ylim((0,255))
     
    # Prepare the cross to be updated later
    vlin = ax1.axvline(x=dataBefore.shape[1]/2)
    hlin = ax1.axhline(y=dataBefore.shape[0]/2)

    #Prepare the plots to be updated later. The generic first point is the central point in the image
     
    imag = ax1.imshow(dataBefore,cmap='gray') #norm=colors.Normalize(vmin=0,vmax=254,clip=True),origin='lower')
    row, = ax2.plot(dataBefore[dataBefore.shape[1]//2,:])
    column, = ax3.plot(dataBefore[:,dataBefore.shape[0]//2])
    # Format the axis
    ax1.set_yticks([dataBefore.shape[0]//2])
    ax1.set_xticks([dataBefore.shape[1]//2])
    ax1.set_title('Line Profile BEFORE Bias Field Correction')
     
    ax2.set_yticks([0, max(dataBefore[dataBefore.shape[0]//2,:]) ])
    ax2.set_title('Horizontal')
    ax2.set_xlim((0,dataBefore.shape[1]//2))
    ax2.set_xticks([])
     
    ax3.set_yticks([0, max(dataBefore[:,dataBefore.shape[0]//2]) ])
    ax3.set_title('Vertical')
    ax3.set_xlim((0,dataBefore.shape[1]//2))
    ax3.set_xticks([])

########## BEGIN FIGURE  AFTER CORRECTION #########
    plt.ion()
    fig_after = plt.figure(figsize=(8.3,3))
    # Page width is 8.3 inches
    # Create two plots: Picture and horizontal profiles
    ax4= fig_after.add_subplot(121)
    ax5= fig_after.add_subplot(143)
    ax6= fig_after.add_subplot(144)
    
     
    # Given that the two figures on the right are very
    # close, we set the labels on the rightmost one
    # to the right
    ax6.yaxis.set_ticks_position('right')
     
    # If the limits of the y axis are not set, the first point will determine
    # the span of it
    ax5.set_ylim((0,255))
    ax6.set_ylim((0,255))
     
    # Prepare the cross to be updated later
    vlin_after = ax4.axvline(x=dataAfter.shape[1]/2)
    hlin_after = ax4.axhline(y=dataAfter.shape[0]/2)

    #Prepare the plots to be updated later. The generic first point is the central point in the image
     
    imag_after = ax4.imshow(dataAfter,cmap='gray') 
    row_after, = ax5.plot(dataAfter[dataAfter.shape[1]//2,:])
    column_after,= ax6.plot(dataAfter[:,dataAfter.shape[0]//2])
    
   # Format the axis
    ax4.set_yticks([dataAfter.shape[0]//2])
    ax4.set_xticks([dataAfter.shape[1]//2])
    ax4.set_title('Line Profile AFTER Bias Field Correction')
     
    ax5.set_yticks([0, max(dataAfter[dataAfter.shape[0]//2,:]) ])
    ax5.set_title('Horizontal')
    ax5.set_xlim((0,dataAfter.shape[1]//2))
    ax5.set_xticks([])
     
    ax6.set_yticks([0, max(dataAfter[:,dataAfter.shape[0]//2]) ])
    ax6.set_title('Vertical')
    ax6.set_xlim((0,dataAfter.shape[1]//2))
    ax6.set_xticks([])
     

    fig3, ax7 = plt.subplots()
    # Plotting both the curves simultaneously
    ax7.plot(dataBefore[dataBefore.shape[1]//2,:],color='r', label='Before')
    #ax8 = ax7.twinx()
    ax7.plot(dataAfter[dataAfter.shape[1]//2,:], color='g', label='After')
    ax7.set_xlabel('Pixel number')
    ax7.set_ylabel('Pixel intensity')

    # Naming the whole graph
    plt.title("Horizontal Line profile before vs after bias correction")

    # Adding legend, which helps us recognize the curve according to it's color
    plt.legend()


    # To load the display window
    plt.show()

    output_folder=r'DATA_ROOT\3_StructuralPreProcessing\resultados_preprocesado_prueba_novale\checking_results_v2' #Writethe path where you want to store your results
    final_fig=fig3.savefig(os.path.join(output_folder,mod,pat,pat+mod+'_LineProfiles.png'))
    plt.show()
    
    return final_fig

In [ ]:
pathLikeBefore=r"DATA_ROOT\3_StructuralPreProcessing\preprocesado_results_v2\FLAIR_3D_SPIR_TRA_IQ\PAT_71_\PAT_71_FLAIR_3D_SPIR_TRA_IQ_cropped.nii.gz"
pathLikeAfter=r"DATA_ROOT\3_StructuralPreProcessing\preprocesado_results_v2\FLAIR_3D_SPIR_TRA_IQ\PAT_71_\PAT_71_FLAIR_3D_SPIR_TRA_IQ_biasCorrected.nii.gz"
pat_number='PAT_71_'
mod='FLAIR_3D_SPIR_TRA_IQ'
cropNeck_temp_jpg=nifti2jpg(pathLikeBefore, pat_number, mod, 'cropped')
NoBias_temp_jpg=nifti2jpg(pathLikeAfter, pat_number, mod, 'biasCorrected')
line_profile=BiasFieldCheck(cropNeck_temp_jpg,NoBias_temp_jpg,pat_number,mod)